In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import itertools
import time
import torch
import pylab as plt
# %matplotlib inline
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm

import memory as mem   
from memory import PrioritizedMemory
from sac import SACAgent

In [ ]:
env_name = 'Pendulum-v1'
# env_name = 'CartPole-v0'

env = gym.make(env_name)
# if isinstance(env.action_space, spaces.Box):
#     env = DiscreteActionWrapper(env,5)

ac_space = env.action_space
o_space = env.observation_space
print(ac_space)
print(o_space)
print(list(zip(env.observation_space.low, env.observation_space.high)))

In [ ]:
max_episodes=600
max_steps=500 
buffer = PrioritizedMemory()
agent = SACAgent(env.observation_space.shape[0], env.action_space.shape[0], max_steps)

In [ ]:
ob,_info = env.reset()
print(ob)
agent.actor(torch.FloatTensor(ob).unsqueeze(0))

In [ ]:
stats = []
losses = []

In [ ]:
for i in range(max_episodes):
    # print("Starting a new episode")    
    total_reward = 0
    ob, _info = env.reset()
    done = False
    for t in range(max_steps):
        with torch.no_grad():
            a, _ = agent.actor.sample(torch.FloatTensor(ob).unsqueeze(0))
        a = a.numpy()[0]

        (ob_new, reward, done, trunc, _info) = env.step(a)
        buffer.add((ob, a, reward, ob_new, float(done)))
        total_reward+= reward
        ob=ob_new        
        agent.update(buffer)
        if done: 
            break    
    stats.append([i,total_reward,t+1])
    # gen.reset()
    
    # if ((i-1)%20==0):
    print("{}: Reward: {}".format(i, total_reward))

In [ ]:
test_stats = []
episodes=50
env_ = env    # without rendering
#env_ = env_eval # with rendering

for i in range(episodes):
    total_reward = 0
    ob, _info = env_.reset()
    for t in range(max_steps):
        done = False
        with torch.no_grad():
            a, _ = agent.actor.sample(torch.FloatTensor(ob).unsqueeze(0))
        a = a.numpy()[0]
        (ob_new, reward, done, trunc, _info) = env_.step(a)
        total_reward+= reward
        ob=ob_new        
        if done: 
            break    
    print(i, "Reward:", total_reward)
    test_stats.append([i,total_reward,t+1])        

In [ ]:
test_stats_np = np.array(test_stats)
print(np.mean(test_stats_np[:,1]), "+-", np.std(test_stats_np[:,1]))